In [18]:
!pip install --no-deps git+https://github.com/mit-han-lab/llm-awq


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Cloning https://github.com/mit-han-lab/llm-awq to /tmp/pip-req-build-r9p5dm31
  Running command git clone --filter=blob:none --quiet https://github.com/mit-han-lab/llm-awq /tmp/pip-req-build-r9p5dm31


  Resolved https://github.com/mit-han-lab/llm-awq to commit d4c12416a88ba251aaa2d75f39ab6df147d01f3c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for awq: filename=awq-0.1.0-py3-none-any.whl size=147043 sha256=1c1b285c34af5673404cc91c7d1ce9cf84843f4f1b1d0fe9937358088338a408
  Stored in directory: /tmp/pip-ephem-wheel-cache-hxmqcyuq/wheels/59/09/c0/315865998c2a3d3df749c8b363a608a0bbd4ed207cd824a138
Successfully built awq

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoConfig, GPT2LMHeadModel, GPT2Tokenizer, AutoModelForCausalLM, AutoTokenizer



In [3]:
model_path = 'tuned_models/rugpt13B'


In [4]:

generation_args = {'max_length': 512,
                   'num_return_sequences': 1,
                   'do_sample': True,
                   'no_repeat_ngram_size': 10,
                   'temperature': 0.8,
                   'top_p': 0.6,
                   'top_k': 0,
                   }



In [5]:

tokenizer = GPT2Tokenizer.from_pretrained(model_path)


In [6]:
class TextDataset(Dataset):
    def __init__(self, path, tokenizer, seq_length=1024):
        with open(path) as f:
            data = f.read()
        tokens = tokenizer.encode(data)
        examples = []
        for i in range(0, len(tokens) - seq_length + 1, seq_length):
            examples.append(tokens[i:i + seq_length])
        self.samples = torch.LongTensor(examples)
        print('Loaded samples:', len(self.samples))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, item):
        return self.samples[item]

In [7]:
valid_path = 'dataset/pelevin_valid.txt'

In [8]:
with open(valid_path) as f:
    data = f.read()
tokens = tokenizer.encode(data)

pbs_k = len(data)/len(tokens)
print(pbs_k)

Token indices sequence length is longer than the specified maximum sequence length for this model (130662 > 2048). Running this sequence through the model will result in indexing errors


3.7976305276208846


In [ ]:
train_dataset = TextDataset('dataset/pelevin_train.txt', tokenizer)
valid_dataset = TextDataset(valid_path, tokenizer)
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False)

In [ ]:
tokenizer.decode(train_dataset[0])

'Annotation\n\n\nВбойщик KGBT+ (автор классических стримов «Катастрофа», «Летитбизм» и других) известен всей планете как титан перформанса и духа. Если вы не слышали его имени, значит, эпоха green power для вас еще не наступила и завоевавшее планету искусство B2B (brain-to-brain streaming) каким-то чудом обошло вас стороной.\n\nНо эта книга – не просто очередное жизнеописание звезды шоу-биза. Это учебник успеха. Великий вбойщик дает множество мемо-советов нацеленному на победу молодому исполнителю. KGBT+ подробно рассказывает историю создания своих шедевров и комментирует сложные факты своей биографии, включая убийства, покушения и почти вековую отсидку в баночной тюрьме, а также опровергает многочисленные слухи о своей личной жизни. Настоящее издание впервые включает повесть «Дом Бахии» о прошлой (предположительно) жизни легендарного вбойщика в Японии и Бирме.\n\nКнига не только подарит вам несколько интересных вечеров, но и познакомит с аутентичными древними психотехниками, применени

In [ ]:
# Step 0: 2.5511 lr = 3e-5 Loss: 2.4204 bpbs: 0.637344

In [ ]:
def mean_loss(model, loader):
    losses = []
    for batch in loader:
        batch = batch.to(model.device)
        with torch.no_grad():
            out = model(batch, labels=batch)
        losses.append(out.loss.item())
    return float(np.mean(losses))

baseline_loss = 2.4204
baseline_loss

2.4204

In [ ]:
from torch.utils.data import DataLoader, Subset
from itertools import islice

# --- pick a lightweight slice ---
#   128 batches × 4 seqs × 512 tokens  ≈ 262 k tokens
cal_subset = Subset(train_dataset, range(0, 128*4))   # first 512 samples


In [ ]:
from datasets import Dataset
cal_ds = Dataset.from_list([{"input_ids": item.tolist()} for item in cal_subset])

In [ ]:
len(cal_ds)

512

In [ ]:
cal_ds

Dataset({
    features: ['input_ids'],
    num_rows: 512
})

In [ ]:
from awq import AutoAWQForCausalLM

from transformers import AutoTokenizer


ImportError: cannot import name 'AutoAWQForCausalLM' from 'awq' (unknown location)

In [ ]:

model_id = "tuned_models/rugpt13B"
save_dir = "rugpt13B-AWQ-4bit"

model = AutoAWQForCausalLM.from_pretrained(model_id, fuse_layers=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# 128 calibration prompts drawn from your Pelevin‑style subset ------------
from random import sample
from datasets import load_dataset
calib = sample(load_dataset("pelevin_dataset", split="train")["text"], 128)

model.quantize(tokenizer, calib, w_bit=4, q_group_size=128)
model.save_quantized(save_dir)
tokenizer.save_pretrained(save_dir)


In [5]:
get_calib_dataset(tokenizer)

NameError: name 'tokenizer' is not defined

In [21]:

from llmcompressor.entrypoints import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = [
    GPTQModifier(
        scheme="W4A16",
        layer_wise=False,             # global quant bias = more stable
        damp_percent=0.01,            # tiny damping helps eos logits
        sequential_update=False,
    )
]

oneshot(
    model="tuned_models/rugpt13B",
    dataset=cal_ds,
    recipe=recipe,
    output_dir="candidates/rpgpt_13b_pelevin_gptq_w4a16",
    max_seq_length=512,
    num_calibration_samples=len(cal_ds),
)


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at tuned_models/rugpt13B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2025-05-03T08:24:45.476995+0000 | reset | INFO - Compression lifecycle reset
2025-05-03T08:24:45.477988+0000 | from_modifiers | INFO - Creating recipe from modifiers


Preparing intermediates cache:   0%|          | 0/512 [00:00<?, ?it/s]
/usr/local/lib/python3.12/dist-packages/llmcompressor/modifiers/quantization/gptq/base.py:202: UserWarning: Falling back to layer_sequential pipeline
  warnings.warn("Falling back to layer_sequential pipeline")
(40/40): Propagating: 100%|██████████| 512/512 [00:00<00:00, 651.55it/s]
manager stage: Modifiers initialized


2025-05-03T08:25:50.619228+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers


manager stage: Modifiers finalized


2025-05-03T08:25:50.621426+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2025-05-03T08:25:50.621707+0000 | save_pretrained_wrapper | INFO - Fetching state_dict - this may take some time
2025-05-03T08:25:55.615368+0000 | save_pretrained_wrapper | INFO - Fetching compressor
2025-05-03T08:25:55.616004+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Quantized Compression: 100%|██████████| 487/487 [00:00<00:00, 570.66it/s]

2025-05-03T08:25:56.475100+0000 | save_pretrained_wrapper | INFO - Saving compressed model to disk


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50272, 5120)
    (wpe): Embedding(2048, 5120)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-39): 40 x GPT2Block(
        (ln_1): LayerNorm((5120,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=15360, nx=5120)
          (c_proj): Conv1D(nf=5120, nx=5120)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((5120,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=20480, nx=5120)
          (c_proj): Conv1D(nf=5120, nx=20480)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((5120,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=5120, out_features=50272, bias=False)
)

In [21]:
from transformers import AutoModelForCausalLM
q_model = AutoModelForCausalLM.from_pretrained(
            "candidates/rpgpt_13b_pelevin_gptq_w4a16",
            device_map="auto", 
            trust_remote_code=True)


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [22]:

quant_loss = mean_loss(q_model, valid_loader)   # reuse your existing loader
print(f"Δloss = {quant_loss - baseline_loss:.4f}")


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Δloss = nan


In [23]:
prompt = 'На словах ты Лев Толстой, а на деле'

input_ids = tokenizer(prompt, return_tensors='pt').input_ids


In [24]:
input_ids

tensor([[ 1487, 14455,   745, 11023, 32060,    17,   365,   310,  2679]])

In [25]:

out_ids = q_model.generate(input_ids=input_ids.to('cuda'),
                         eos_token_id=tokenizer.eos_token_id,
                         **generation_args).tolist()


/opt/pytorch/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:112: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:

prompt_len = len(input_ids[0])
for seq in out_ids:
    output = tokenizer.decode(seq)

    if '</s>' in output:
        output = output[:output.find('</s>')].strip()

    text = output
    print('-'*80)
    print(text)

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, os

model_path = "candidates/rpgpt_13b_pelevin_gptq_w4a16"

tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
            model_path, device_map="auto", trust_remote_code=True)

prompt = "На словах ты Лев Толстой, а на деле"
enc = tok(prompt, return_tensors="pt").to("cuda")

# ---- sanity check ----------------------------------------------------
print("max token id:", enc.input_ids.max().item(),
      "vocab size:", model.config.vocab_size)
assert enc.input_ids.max() < model.config.vocab_size, "bad tokenizer!"
# ---------------------------------------------------------------------


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

max token id: 32060 vocab size: 50272


In [19]:
import torch, os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

model_path = "candidates/rpgpt_13b_pelevin_gptq_w4a16"

tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
            model_path, device_map="auto",
            trust_remote_code=True, torch_dtype="auto")

inputs = tok("ping", return_tensors="pt").to("cuda")
with torch.no_grad():
    logits = model(**inputs).logits
print("it works:", logits.shape)


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

it works: torch.Size([1, 1, 50272])


In [21]:
gen_out = model.generate(
    input_ids.to(model.device),
    max_new_tokens=64,     # keep it short first
    do_sample=False        # greedy = no sampling kernels yet
)
print(tok.decode(gen_out[0], skip_special_tokens=True))


На словах ты Лев Толстой, а на деле


In [22]:
prompt = "На словах ты Лев Толстой, а на деле"

ids = tok(prompt, add_special_tokens=False,
          return_tensors="pt").input_ids.to(model.device)


In [23]:
ids

tensor([[ 1487, 14455,   745, 11023, 32060,    17,   365,   310,  2679]],
       device='cuda:0')

In [25]:
gen_out = model.generate(
    input_ids.to(model.device),
    max_new_tokens=64,     # keep it short first
    do_sample=False        # greedy = no sampling kernels yet
)
print(tok.decode(gen_out[0], skip_special_tokens=True))

gen_out[0]

На словах ты Лев Толстой, а на деле


tensor([ 1487, 14455,   745, 11023, 32060,    17,   365,   310,  2679,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0], device='cuda:0')

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "candidates/rpgpt_13b_pelevin_gptq_w4a16"
tok   = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path,
                                             device_map="auto",
                                             trust_remote_code=True)
model.eval()

prompt = "На словах ты Лев Толстой, а на деле"
ids    = tok(prompt, add_special_tokens=False,
             return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    # we only need the logits for the *next* token → take last hidden‑state
    next_logits = model(ids).logits[0, -1]          # shape (vocab_size,)

topk = torch.topk(next_logits, 10)
print("top‑10 token ids:", topk.indices.tolist())
print("top‑10 logits   :", topk.values.tolist()[:3], "…")
print("eos token id    :", tok.eos_token_id)
print("eos log‑prob rank:",
      (next_logits > next_logits[tok.eos_token_id]).sum().item())


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

top‑10 token ids: [7, 6, 4, 5, 1, 0, 2, 3, 8, 9]
top‑10 logits   : [0.0, 0.0, 0.0] …
eos token id    : 3
eos log‑prob rank: 0


In [31]:
for l in next_logits: 
    if l>0: print(l)